# Amazon Retail Demand Forecasting
### Store-level Beverage Sales | Time-Series Forecasting

**Author:** Abdelaziz Amr Mettwally

## Project overview
This project explores daily beverage sales for Store 1 and compares forecasting approaches using a chronological holdout set. The workflow includes exploratory analysis, time-based feature engineering, baseline and machine-learning models, and evaluation with MAE and RMSE.

> **Data note:** The PDF supplied for reconstruction shows a dataset with `date`, `store_nbr`, `family`, `sales`, and `onpromotion` columns. It does not include the underlying CSV. Download the training data from Kaggle's *Store Sales - Time Series Forecasting* competition and set `DATA_PATH` below. The original PDF's reported scores are not hard-coded as results; this notebook calculates metrics from the data you run it on.


## 1. Setup

Install any missing packages before running the notebook. XGBoost is optional; if it is unavailable, the notebook still runs the Random Forest and seasonal-naive baseline.

In [ ]:
# If needed, run this in a notebook cell:
# %pip install pandas numpy matplotlib scikit-learn statsmodels xgboost

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 2. Load data

Download `train.csv` from the Kaggle competition and point `DATA_PATH` to it. If your file is already named `sales.csv`, that is fine too.

In [ ]:
DATA_PATH = Path("../data/train.csv")  # Change this path to your CSV file

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Download train.csv from Kaggle "
        "Store Sales - Time Series Forecasting and update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
required_columns = {"date", "store_nbr", "family", "sales"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head())


## 3. Select a store and product family

To match the analysis in the supplied report, the default example uses Store 1 and the `BEVERAGES` product family. Change these values to explore another segment.

In [ ]:
STORE_NBR = 1
PRODUCT_FAMILY = "BEVERAGES"

segment = df.loc[
    (df["store_nbr"] == STORE_NBR) & (df["family"] == PRODUCT_FAMILY),
    ["date", "sales"] + (["onpromotion"] if "onpromotion" in df.columns else [])
].copy()

if segment.empty:
    raise ValueError("No rows found for the selected store and product family.")

segment = (
    segment.groupby("date", as_index=False)
    .agg(sales=("sales", "sum"))
    .sort_values("date")
    .set_index("date")
)

# Create a continuous daily index. Missing dates are left as missing at this stage
# so the handling decision is explicit rather than silently interpolated.
full_index = pd.date_range(segment.index.min(), segment.index.max(), freq="D")
segment = segment.reindex(full_index)
segment.index.name = "date"

print(f"Selected segment: Store {STORE_NBR} / {PRODUCT_FAMILY}")
print(f"Date range: {segment.index.min().date()} to {segment.index.max().date()}")
print(f"Missing daily sales values: {segment['sales'].isna().sum():,}")
display(segment.head())


## 4. Exploratory data analysis

Inspect the series and its weekly pattern before fitting models. Missing daily values are filled with zero here to maintain a continuous daily series; this is an assumption and should be revisited if missing rows mean unavailable data rather than zero sales.

In [ ]:
series = segment["sales"].asfreq("D").fillna(0.0)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(series.index, series.values, linewidth=0.8)
ax.set_title(f"Daily Sales — Store {STORE_NBR} — {PRODUCT_FAMILY}")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(series.describe().to_frame("sales"))


## 5. Feature engineering

All lag and rolling features use only current/past observations available at the forecast origin. Rolling statistics are shifted by one day to avoid including the target day's sales in its own predictors.

In [ ]:
def make_features(y: pd.Series) -> pd.DataFrame:
    data = pd.DataFrame(index=y.index)
    data["sales"] = y
    data["lag_1"] = y.shift(1)
    data["lag_7"] = y.shift(7)
    data["lag_14"] = y.shift(14)

    prior_sales = y.shift(1)
    data["rolling_mean_7"] = prior_sales.rolling(7).mean()
    data["rolling_std_7"] = prior_sales.rolling(7).std()
    data["rolling_mean_14"] = prior_sales.rolling(14).mean()
    data["rolling_std_14"] = prior_sales.rolling(14).std()

    data["day_of_week"] = data.index.dayofweek
    data["day"] = data.index.day
    data["month"] = data.index.month
    data["week_of_year"] = data.index.isocalendar().week.astype(int)
    data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)
    return data

model_data = make_features(series).dropna()
feature_columns = [c for c in model_data.columns if c != "sales"]
X = model_data[feature_columns]
y = model_data["sales"]

print(f"Rows available for modeling: {len(model_data):,}")
display(model_data.head())


## 6. Chronological train/test split

The last 20% of observations are held out as test data. No random shuffle is used because time order matters in forecasting.

In [ ]:
split_idx = int(len(model_data) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print("Training period:", y_train.index.min().date(), "to", y_train.index.max().date())
print("Testing period: ", y_test.index.min().date(), "to", y_test.index.max().date())
print("Train rows:", len(X_train), "| Test rows:", len(X_test))


## 7. Baseline: seasonal naïve forecast

A useful forecasting project should compare complex models against a simple baseline. The seasonal-naïve prediction uses sales from seven days earlier.

In [ ]:
baseline_pred = y.shift(7).reindex(y_test.index)

def regression_metrics(actual, predicted):
    valid = actual.notna() & pd.Series(predicted, index=actual.index).notna()
    actual_valid = actual.loc[valid]
    pred_valid = pd.Series(predicted, index=actual.index).loc[valid]
    return {
        "MAE": mean_absolute_error(actual_valid, pred_valid),
        "RMSE": np.sqrt(mean_squared_error(actual_valid, pred_valid)),
    }

results = []
baseline_metrics = regression_metrics(y_test, baseline_pred)
results.append({"Model": "Seasonal Naive (7-day)", **baseline_metrics})
baseline_metrics


## 8. Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_pred = pd.Series(rf.predict(X_test), index=y_test.index)

results.append({"Model": "Random Forest", **regression_metrics(y_test, rf_pred)})
print(results[-1])

rf_importance = pd.Series(rf.feature_importances_, index=feature_columns).sort_values()
rf_importance.plot(kind="barh", figsize=(8, 5), title="Random Forest Feature Importance")
plt.tight_layout()
plt.show()


## 9. XGBoost Regressor (optional)

XGBoost is used if installed. Parameters here are a reasonable starting point, not the result of an exhaustive hyperparameter search.

In [ ]:
xgb_pred = None
try:
    from xgboost import XGBRegressor

    xgb = XGBRegressor(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb.fit(X_train, y_train)
    xgb_pred = pd.Series(xgb.predict(X_test), index=y_test.index)
    results.append({"Model": "XGBoost", **regression_metrics(y_test, xgb_pred)})
    print(results[-1])
except ImportError:
    print("XGBoost is not installed; skipping this model. Install it with: %pip install xgboost")


## 10. Time-series cross-validation

Use expanding-window splits on the training period only. Each validation fold is later in time than its training fold. This provides a more robust view than relying on one split alone.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    fold_model = RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    fold_model.fit(X_tr, y_tr)
    fold_pred = fold_model.predict(X_val)

    cv_rows.append({
        "Fold": fold,
        "Validation start": y_val.index.min().date(),
        "Validation end": y_val.index.max().date(),
        "MAE": mean_absolute_error(y_val, fold_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, fold_pred)),
    })

cv_results = pd.DataFrame(cv_rows)
display(cv_results)
print("Mean CV MAE:", round(cv_results["MAE"].mean(), 3))
print("Mean CV RMSE:", round(cv_results["RMSE"].mean(), 3))


## 11. Compare models on the same holdout period

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.style.format({"MAE": "{:.2f}", "RMSE": "{:.2f}"}))

ax = results_df.set_index("Model")[["MAE", "RMSE"]].plot(
    kind="bar", figsize=(10, 5), title="Model Comparison on Chronological Test Set"
)
ax.set_ylabel("Error (lower is better)")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Actual vs predicted

In [ ]:
predictions = {"Seasonal Naive (7-day)": baseline_pred, "Random Forest": rf_pred}
if xgb_pred is not None:
    predictions["XGBoost"] = xgb_pred

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test.index, y_test.values, label="Actual", linewidth=1.5)
for label, pred in predictions.items():
    ax.plot(pred.index, pred.values, label=label, alpha=0.85)
ax.set_title("Actual vs Predicted Sales — Test Period")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 13. Forecasting beyond the observed data

For a true multi-step forecast, future lag features must be built recursively from predictions because actual future sales are unavailable. This example uses the fitted Random Forest and the most recent observed history. Treat the output as an illustrative forecast, not a production-ready business forecast.

In [ ]:
def forecast_recursive(history: pd.Series, model, feature_cols, horizon=30):
    history = history.astype(float).copy()
    future_dates = pd.date_range(history.index.max() + pd.Timedelta(days=1), periods=horizon, freq="D")
    forecast_values = []

    for date in future_dates:
        temp = pd.concat([history, pd.Series(index=[date], dtype=float)])
        row = {
            "lag_1": history.iloc[-1],
            "lag_7": history.iloc[-7] if len(history) >= 7 else np.nan,
            "lag_14": history.iloc[-14] if len(history) >= 14 else np.nan,
            "rolling_mean_7": history.iloc[-7:].mean() if len(history) >= 7 else np.nan,
            "rolling_std_7": history.iloc[-7:].std() if len(history) >= 7 else np.nan,
            "rolling_mean_14": history.iloc[-14:].mean() if len(history) >= 14 else np.nan,
            "rolling_std_14": history.iloc[-14:].std() if len(history) >= 14 else np.nan,
            "day_of_week": date.dayofweek,
            "day": date.day,
            "month": date.month,
            "week_of_year": int(date.isocalendar().week),
            "is_weekend": int(date.dayofweek >= 5),
        }
        X_next = pd.DataFrame([row], index=[date])[feature_cols]
        prediction = float(model.predict(X_next)[0])
        prediction = max(0.0, prediction)
        forecast_values.append(prediction)
        history.loc[date] = prediction

    return pd.Series(forecast_values, index=future_dates, name="forecast")

future_forecast = forecast_recursive(series, rf, feature_columns, horizon=30)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(series.index[-90:], series.iloc[-90:], label="Recent actual sales")
ax.plot(future_forecast.index, future_forecast.values, label="30-day forecast")
ax.set_title("Illustrative 30-Day Recursive Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(future_forecast.to_frame())


## 14. Findings and limitations

Complete this section after running the notebook and reviewing the generated metrics and plots.

- Compare the best-performing model with the seasonal-naïve baseline.
- Describe whether weekly seasonality is visible in the selected series.
- Discuss error magnitude in the context of typical daily sales.
- Note that results are for one store and one product family, not all Amazon retail sales.
- Explain that zero-filling missing dates is an assumption that may affect results.
- Avoid claiming general superiority of one model from a single segment or holdout period.

## Reproducibility

Record the dataset source, date downloaded, package versions, and any parameter changes before publishing. Do not include credentials or private files in the repository.
